# this is the quantization bock
#  this block   get the input from  activation block      
# this block will decide what output we have to store  in the scratch pad
# we have to store the  int 8  quantized value   or 32 bit high precission value 

input output  matrix   :  from activation 
      output_valid 
      is_quantization      this is from control block  
      shift size    :: this  is also from control  we took it  from  cpu  
      input ready signal this is from scratch pad    to store the data scratch pad should ready for it  



 output   output_matrix  ::  to store it into the scratch pad  where to store is decided by 
                             control and memory management 
          output valid :  valid signal 
          ready signal :  this is for valid ready protocall   that we have to 
          


In [1]:
class QuantizationBlock:
    def __init__(self, size):
        """
        size : tile dimension T (matrix is T×T)
        """
        self.size = size

    def run(self, input_matrix, output_valid_in, is_quantization, shift_size, scratchpad_ready):
        """
        Quantization Block Golden Model

        Inputs:
            input_matrix      : T×T matrix (32-bit high precision) from activation block
            output_valid_in   : 1 if input_matrix is valid this cycle
            is_quantization   : 1 = quantize to int8
                                0 = pass through as 32-bit high precision
            shift_size        : number of bits to right shift (from CPU via control block)
            scratchpad_ready  : 1 if scratchpad is ready to accept data

        Outputs:
            output_matrix     : T×T result — int8 or 32-bit depending on is_quantization
            output_valid      : 1 if output_matrix is valid this cycle
            ready             : 1 if this block is ready to accept new input
                                (goes low when scratchpad is not ready)
        """
        T = self.size

        output_matrix = [[0] * T for _ in range(T)]
        output_valid  = 0
        ready         = 0

        # ready follows scratchpad — if scratchpad is busy, we stall
        if scratchpad_ready:
            ready = 1
        else:
            # stall — cannot accept or send data
            return output_matrix, output_valid, ready

        # process only if input is valid
        if output_valid_in:

            if is_quantization == 1:
                # ---- INT8 PATH ----
                # Step 1 : arithmetic right shift by shift_size
                # Step 2 : clip to int8 range [-128, 127]
                for i in range(T):
                    for j in range(T):
                        val = input_matrix[i][j]

                        # arithmetic right shift — preserves sign
                        shifted = int(val) >> shift_size

                        # clip to int8 range
                        clipped = max(-128, min(127, shifted))

                        output_matrix[i][j] = clipped

            else:
                # ---- HIGH PRECISION PATH (32-bit pass-through) ----
                # is_quantization=0 : pass raw 32-bit values as-is
                for i in range(T):
                    for j in range(T):
                        output_matrix[i][j] = input_matrix[i][j]

            output_valid = 1

        return output_matrix, output_valid, ready





import random

T = 4
quant = QuantizationBlock(size=T)

# sample 32-bit input from activation block
input_matrix = [
    [5120,  -3072,  8192,  -512],
    [2048,   1024, -4096,  6144],
    [-8192,  256,   512,  -128],
    [16384, -2048,  4096, -6144]
]

print("=" * 45)
print("TEST 1 — INT8 quantization (shift_size=6)")
print("=" * 45)
out, valid, ready = quant.run(
    input_matrix    = input_matrix,
    output_valid_in = 1,
    is_quantization = 1,
    shift_size      = 6,           # divide by 64 effectively
    scratchpad_ready= 1
)
print("output_matrix:")
for row in out:
    print(row)
print(f"output_valid : {valid}")
print(f"ready        : {ready}")

print()
print("=" * 45)
print("TEST 2 — 32-bit pass-through (is_quantization=0)")
print("=" * 45)
out, valid, ready = quant.run(
    input_matrix    = input_matrix,
    output_valid_in = 1,
    is_quantization = 0,
    shift_size      = 6,           # ignored in this path
    scratchpad_ready= 1
)
print("output_matrix:")
for row in out:
    print(row)
print(f"output_valid : {valid}")
print(f"ready        : {ready}")

print()
print("=" * 45)
print("TEST 3 — Scratchpad NOT ready (stall)")
print("=" * 45)
out, valid, ready = quant.run(
    input_matrix    = input_matrix,
    output_valid_in = 1,
    is_quantization = 1,
    shift_size      = 6,
    scratchpad_ready= 0            # scratchpad busy → stall
)
print("output_matrix:")
for row in out:
    print(row)
print(f"output_valid : {valid}")
print(f"ready        : {ready}")

TEST 1 — INT8 quantization (shift_size=6)
output_matrix:
[80, -48, 127, -8]
[32, 16, -64, 96]
[-128, 4, 8, -2]
[127, -32, 64, -96]
output_valid : 1
ready        : 1

TEST 2 — 32-bit pass-through (is_quantization=0)
output_matrix:
[5120, -3072, 8192, -512]
[2048, 1024, -4096, 6144]
[-8192, 256, 512, -128]
[16384, -2048, 4096, -6144]
output_valid : 1
ready        : 1

TEST 3 — Scratchpad NOT ready (stall)
output_matrix:
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
output_valid : 0
ready        : 0
